2.1

In [7]:
print("""
给定字符序列 "ababc"，词汇表 V = {a, b, c}，一阶马尔可夫模型 p(x_t | x_{t-1})，拉普拉斯平滑（加1平滑）。

统计转移频次（从 t-1 到 t）：
序列 ababc 中相邻对：(a,b), (b,a), (a,b), (b,c)
  从 a -> b: 2次
  从 b -> a: 1次
  从 b -> c: 1次
  其他转移: 0次

拉普拉斯平滑公式：
p(x_t = v | x_{t-1} = u) = (count(u -> v) + 1) / (count(u -> *) + |V|)

其中 count(u -> *) = 从 u 出发的所有转移次数之和（在序列中）。

从 b 出发的转移：b->a 1次，b->c 1次，总次数 = 2。|V| = 3。

1. p(a | b) = (count(b->a) + 1) / (count(b->*) + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

2. p(c | b) = (count(b->c) + 1) / (count(b->*) + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

答案：
p(a|b) = 2/5 = 0.4
p(c|b) = 2/5 = 0.4
""")


给定字符序列 "ababc"，词汇表 V = {a, b, c}，一阶马尔可夫模型 p(x_t | x_{t-1})，拉普拉斯平滑（加1平滑）。

统计转移频次（从 t-1 到 t）：
序列 ababc 中相邻对：(a,b), (b,a), (a,b), (b,c)
  从 a -> b: 2次
  从 b -> a: 1次
  从 b -> c: 1次
  其他转移: 0次

拉普拉斯平滑公式：
p(x_t = v | x_{t-1} = u) = (count(u -> v) + 1) / (count(u -> *) + |V|)

其中 count(u -> *) = 从 u 出发的所有转移次数之和（在序列中）。

从 b 出发的转移：b->a 1次，b->c 1次，总次数 = 2。|V| = 3。

1. p(a | b) = (count(b->a) + 1) / (count(b->*) + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

2. p(c | b) = (count(b->c) + 1) / (count(b->*) + 3) = (1 + 1) / (2 + 3) = 2/5 = 0.4

答案：
p(a|b) = 2/5 = 0.4
p(c|b) = 2/5 = 0.4



2.2

In [2]:
import re
from collections import Counter

def preprocess_text(text, n):
    
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text_lower = text.lower()
    # 只保留字母和空格
    text_clean = re.sub(r'[^a-z\s]', '', text_lower)
    
    # 2. 按空格分词，并过滤空字符串
    tokens = [word for word in text_clean.split() if word]
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    # 按频率降序排序，频率相同则按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签
    features = []
    labels = []
    for i in range(len(tokens) - n):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n])
    
    return vocab, features, labels


text = "The time machine"
n = 2
vocab, features, labels = preprocess_text(text, n)

print(f"输入文本: '{text}'")
print(f"n = {n}")
print(f"词汇表: {vocab}")
print(f"特征列表: {features}")
print(f"标签列表: {labels}")

# 额外测试
print("\n--- 额外测试 ---")
text2 = "Hello world! This is a test. Test test."
n = 3
vocab2, features2, labels2 = preprocess_text(text2, n)
print(f"输入文本: '{text2}'")
print(f"n = {n}")
print(f"词汇表: {vocab2}")
print(f"特征列表: {features2}")
print(f"标签列表: {labels2}")

输入文本: 'The time machine'
n = 2
词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time']]
标签列表: ['machine']

--- 额外测试 ---
输入文本: 'Hello world! This is a test. Test test.'
n = 3
词汇表: {'test': 0, 'a': 1, 'hello': 2, 'is': 3, 'this': 4, 'world': 5}
特征列表: [['hello', 'world', 'this'], ['world', 'this', 'is'], ['this', 'is', 'a'], ['is', 'a', 'test'], ['a', 'test', 'test']]
标签列表: ['is', 'a', 'test', 'test', 'test']


3.1

In [13]:
print("""
考虑线性RNN（无偏置）：
    h_t = W_hh * h_{t-1} + W_hx * x_t
    o_t = W_oh * h_t
损失函数：L = 1/2 * Σ_{t=1}^{T} (o_t - y_t)^2

通过时间反向传播（BPTT），展开所有时间步：
    ∂L/∂W_hh = Σ_{t=1}^{T} ∂L_t/∂W_hh

对于每个时间步 t：
    ∂L_t/∂W_hh = (∂L_t/∂o_t) * (∂o_t/∂h_t) * (∂h_t/∂W_hh)
    
其中：
    ∂L_t/∂o_t = o_t - y_t
    ∂o_t/∂h_t = W_oh^T

∂h_t/∂W_hh 需要展开所有之前的时间步：
    h_t = W_hh * h_{t-1} + W_hx * x_t
    ∂h_t/∂W_hh = h_{t-1}^T + W_hh * (∂h_{t-1}/∂W_hh)

展开递归得到：
    ∂h_t/∂W_hh = Σ_{k=1}^{t} (W_hh)^{t-k} * h_{k-1}^T

因此最终梯度表达式为：
    ∂L/∂W_hh = Σ_{t=1}^{T} (o_t - y_t) * W_oh^T * Σ_{k=1}^{t} (W_hh)^{t-k} * h_{k-1}^T

梯度消失或爆炸的条件：
    - 当 ||W_hh|| < 1 时，随着 t-k 增大，(W_hh)^{t-k} 趋近于0，导致梯度消失
    - 当 ||W_hh|| > 1 时，随着 t-k 增大，(W_hh)^{t-k} 趋向无穷，导致梯度爆炸
    - 特别地，当 W_hh 的最大特征值 λ_max < 1 时梯度消失，λ_max > 1 时梯度爆炸
""")


考虑线性RNN（无偏置）：
    h_t = W_hh * h_{t-1} + W_hx * x_t
    o_t = W_oh * h_t
损失函数：L = 1/2 * Σ_{t=1}^{T} (o_t - y_t)^2

通过时间反向传播（BPTT），展开所有时间步：
    ∂L/∂W_hh = Σ_{t=1}^{T} ∂L_t/∂W_hh

对于每个时间步 t：
    ∂L_t/∂W_hh = (∂L_t/∂o_t) * (∂o_t/∂h_t) * (∂h_t/∂W_hh)

其中：
    ∂L_t/∂o_t = o_t - y_t
    ∂o_t/∂h_t = W_oh^T

∂h_t/∂W_hh 需要展开所有之前的时间步：
    h_t = W_hh * h_{t-1} + W_hx * x_t
    ∂h_t/∂W_hh = h_{t-1}^T + W_hh * (∂h_{t-1}/∂W_hh)

展开递归得到：
    ∂h_t/∂W_hh = Σ_{k=1}^{t} (W_hh)^{t-k} * h_{k-1}^T

因此最终梯度表达式为：
    ∂L/∂W_hh = Σ_{t=1}^{T} (o_t - y_t) * W_oh^T * Σ_{k=1}^{t} (W_hh)^{t-k} * h_{k-1}^T

梯度消失或爆炸的条件：
    - 当 ||W_hh|| < 1 时，随着 t-k 增大，(W_hh)^{t-k} 趋近于0，导致梯度消失
    - 当 ||W_hh|| > 1 时，随着 t-k 增大，(W_hh)^{t-k} 趋向无穷，导致梯度爆炸
    - 特别地，当 W_hh 的最大特征值 λ_max < 1 时梯度消失，λ_max > 1 时梯度爆炸



3.2

In [9]:
import numpy as np
import sys

np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize, precision=8, suppress=False, edgeitems=1000000)

def rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h):
    # 线性变换
    a_t = np.dot(h_prev, W_hh.T) + np.dot(x_t, W_hx.T) + b_h
    # tanh激活
    h_t = np.tanh(a_t)
    
    cache = (x_t, h_prev, W_hh, W_hx, b_h, a_t, h_t)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    x_t, h_prev, W_hh, W_hx, b_h, a_t, h_t = cache
    
    # tanh的导数: dtanh/da = 1 - tanh^2
    da_t = dh_next * (1 - h_t ** 2)
    
    # 对 b_h 的梯度
    db_h = np.sum(da_t, axis=0)
    
    # 对 W_hx 的梯度
    dW_hx = np.dot(da_t.T, x_t)
    
    # 对 W_hh 的梯度
    dW_hh = np.dot(da_t.T, h_prev)
    
    # 对 h_prev 的梯度
    dh_prev = np.dot(da_t, W_hh)
    
    # 对 x_t 的梯度
    dx_t = np.dot(da_t, W_hx)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db_h

def print_array_full(arr, name):
    """完整打印数组，不使用省略号"""
    print(f"{name}:")
    print(arr.tolist())

# 设置参数
batch_size = 2
input_size = 4
hidden_size = 3

# 随机初始化
np.random.seed(42)
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
b_h = np.random.randn(hidden_size)

print_array_full(x_t, "输入 x_t")
print_array_full(h_prev, "上一隐藏状态 h_prev")
print_array_full(W_hh, "权重 W_hh")
print_array_full(W_hx, "权重 W_hx")
print_array_full(b_h, "偏置 b_h")

# 前向传播
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h)
print_array_full(h_t, "前向传播结果 h_t")

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size)
print_array_full(dh_next, "上游梯度 dh_next")

dx_t, dh_prev, dW_hh, dW_hx, db_h = rnn_cell_backward(dh_next, cache)

print("\n反向传播结果:")
print_array_full(dx_t, "dx_t")
print_array_full(dh_prev, "dh_prev")
print_array_full(dW_hh, "dW_hh")
print_array_full(dW_hx, "dW_hx")
print_array_full(db_h, "db_h")

输入 x_t:
[[0.4967141530112327, -0.13826430117118466, 0.6476885381006925, 1.5230298564080254], [-0.23415337472333597, -0.23413695694918055, 1.5792128155073915, 0.7674347291529088]]
上一隐藏状态 h_prev:
[[-0.4694743859349521, 0.5425600435859647, -0.46341769281246226], [-0.46572975357025687, 0.24196227156603412, -1.913280244657798]]
权重 W_hh:
[[-1.7249178325130328, -0.5622875292409727, -1.0128311203344238], [0.3142473325952739, -0.9080240755212109, -1.4123037013352915], [1.465648768921554, -0.22577630048653566, 0.06752820468792384]]
权重 W_hx:
[[-1.4247481862134568, -0.5443827245251827, 0.11092258970986608, -1.1509935774223028], [0.37569801834567196, -0.600638689918805, -0.2916937497932768, -0.6017066122293969], [1.8522781845089378, -0.013497224737933921, -1.0577109289559004, 0.822544912103189]]
偏置 b_h:
[-1.2208436499710222, 0.2088635950047554, -1.9596701238797756]
前向传播结果 h_t:
[[-0.9881267002788087, -0.5458992324599519, -0.8647638146116067], [0.8134714505875482, 0.932233004490598, -0.99962871733333

4.1


In [20]:
print("""
深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。

参数计算（包括所有全连接层的权重和偏置，忽略嵌入层和输出层之前的投影）：

1. 第一层（前向 + 后向）：
   - 输入到隐藏权重：前向 D*H，后向 D*H，共 2*D*H
   - 输入到隐藏偏置：前向 H，后向 H，共 2*H
   - 隐藏到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 隐藏到隐藏偏置：前向 H，后向 H，共 2*H
   小计：2*D*H + 2*H*H + 4*H

2. 中间层（第2层到第L层），每层（前向 + 后向）：
   - 输入到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 输入到隐藏偏置：前向 H，后向 H，共 2*H
   - 隐藏到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 隐藏到隐藏偏置：前向 H，后向 H，共 2*H
   小计每层：4*H*H + 4*H

3. 所有层总计：
   - 第1层：2*D*H + 2*H*H + 4*H
   - 第2到L层（共L-1层）：(L-1) * (4*H*H + 4*H)

总参数 = 2*D*H + 2*H*H + 4*H + (L-1)*4*H*H + (L-1)*4*H
       = 2*D*H + (2 + 4*L - 4)*H*H + (4 + 4*L - 4)*H
       = 2*D*H + (4*L - 2)*H*H + 4*L*H

最终表达式：
    Total = 2*D*H + 4*L*H + (4*L - 2)*H^2
""")


深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。

参数计算（包括所有全连接层的权重和偏置，忽略嵌入层和输出层之前的投影）：

1. 第一层（前向 + 后向）：
   - 输入到隐藏权重：前向 D*H，后向 D*H，共 2*D*H
   - 输入到隐藏偏置：前向 H，后向 H，共 2*H
   - 隐藏到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 隐藏到隐藏偏置：前向 H，后向 H，共 2*H
   小计：2*D*H + 2*H*H + 4*H

2. 中间层（第2层到第L层），每层（前向 + 后向）：
   - 输入到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 输入到隐藏偏置：前向 H，后向 H，共 2*H
   - 隐藏到隐藏权重：前向 H*H，后向 H*H，共 2*H*H
   - 隐藏到隐藏偏置：前向 H，后向 H，共 2*H
   小计每层：4*H*H + 4*H

3. 所有层总计：
   - 第1层：2*D*H + 2*H*H + 4*H
   - 第2到L层（共L-1层）：(L-1) * (4*H*H + 4*H)

总参数 = 2*D*H + 2*H*H + 4*H + (L-1)*4*H*H + (L-1)*4*H
       = 2*D*H + (2 + 4*L - 4)*H*H + (4 + 4*L - 4)*H
       = 2*D*H + (4*L - 2)*H*H + 4*L*H

最终表达式：
    Total = 2*D*H + 4*L*H + (4*L - 2)*H^2



4.2

In [10]:
import torch
import torch.nn as nn
import sys

torch.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize, precision=8, sci_mode=False, edgeitems=1000000)

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, rnn_type='rnn'):
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN
        if rnn_type == 'rnn':
            self.rnn = nn.RNN(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
        else:
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=False,
                bidirectional=True
            )
    
    def forward(self, X):
        # 前向传播
        outputs, h_n = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        if isinstance(self.rnn, nn.LSTM):
            h_n = h_n[0]
        
        # 取最后一层
        last_layer_h = h_n[-2:, :, :]
        # 拼接前向和后向
        final_state = torch.cat([last_layer_h[0], last_layer_h[1]], dim=-1)
        
        return outputs, final_state

def print_tensor_full(tensor, name):
    print(f"{name}:")
    print(tensor.tolist())

# 设置参数
seq_len = 5
batch = 3
input_dim = 4
hidden_dim = 6

# 创建输入
torch.manual_seed(42)
X = torch.randn(seq_len, batch, input_dim)

print_tensor_full(X, "输入 X")

# 创建编码器
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers=2, rnn_type='rnn')

# 前向传播
outputs, final_state = encoder(X)

print(f"\n输入形状: {X.shape}")
print(f"输出形状: {outputs.shape}")
print(f"最终状态形状: {final_state.shape}")
print_tensor_full(outputs[:, 0, :], "outputs (所有时间步, 第1个样本)")
print_tensor_full(final_state, "final_state (所有样本)")

# 验证形状
assert outputs.shape == (seq_len, batch, 2 * hidden_dim)
assert final_state.shape == (batch, 2 * hidden_dim)
print("\n形状验证通过！")

输入 X:
[[[1.9269152879714966, 1.4872840642929077, 0.9007171988487244, -2.1055209636688232], [0.6784183979034424, -1.2345448732376099, -0.04306747764348984, -1.6046669483184814], [-0.7521352767944336, 1.6487230062484741, -0.3924786448478699, -1.4036071300506592]], [[-0.7278813123703003, -0.5594301819801331, -0.7688388824462891, 0.7624453902244568], [1.6423169374465942, -0.1595974713563919, -0.4973975419998169, 0.439589262008667], [-0.7581311464309692, 1.078317642211914, 0.8008005619049072, 1.680620551109314]], [[1.27912437915802, 1.2964228391647339, 0.610466480255127, 1.334737777709961], [-0.2316243201494217, 0.041759490966796875, -0.2515752911567688, 0.859858512878418], [-1.3846737146377563, -0.8712361454963684, -0.223365917801857, 1.7173614501953125]], [[0.3188803195953369, -0.42451897263526917, 0.3057209253311157, -0.7745925188064575], [-1.5575724840164185, 0.9956361055374146, -0.8797858357429504, -0.6011420488357544], [0.36724865436553955, 0.17541083693504333, 1.3851605653762817, -0.

5.1

In [15]:
print("""
Skip-gram模型，中心词 w_c，上下文词 w_o，负采样K个负样本。

词向量表示：
    - v_c: 中心词 w_c 的输入向量
    - u_o: 上下文词 w_o 的输出向量
    - u_{n_k}: 第k个负样本词 n_k 的输出向量

负采样的损失函数（对数似然）：

L = -log σ(u_o^T * v_c) - Σ_{k=1}^{K} log σ(-u_{n_k}^T * v_c)

其中 σ(x) = 1 / (1 + exp(-x)) 是sigmoid函数。

完整目标函数（最大化对数似然，即最小化负对数似然）：

J = -log σ(u_o^T * v_c) - Σ_{k=1}^{K} log σ(-u_{n_k}^T * v_c)

负样本采样方法：
    从噪声分布 P_n(w) 中采样 K 个负样本。
    常用的噪声分布是 unigram 分布的 3/4 次方：
        P_n(w) = (count(w)^{3/4}) / Σ_{i=1}^{V} (count(w_i)^{3/4})
    
    采样时，会从词汇表中按照上述分布独立采样 K 个词（不包括中心词 w_c）。
    实际实现中，通常使用随机采样的方式，每个词被选中的概率与其频率的3/4次方成正比。
""")


Skip-gram模型，中心词 w_c，上下文词 w_o，负采样K个负样本。

词向量表示：
    - v_c: 中心词 w_c 的输入向量
    - u_o: 上下文词 w_o 的输出向量
    - u_{n_k}: 第k个负样本词 n_k 的输出向量

负采样的损失函数（对数似然）：

L = -log σ(u_o^T * v_c) - Σ_{k=1}^{K} log σ(-u_{n_k}^T * v_c)

其中 σ(x) = 1 / (1 + exp(-x)) 是sigmoid函数。

完整目标函数（最大化对数似然，即最小化负对数似然）：

J = -log σ(u_o^T * v_c) - Σ_{k=1}^{K} log σ(-u_{n_k}^T * v_c)

负样本采样方法：
    从噪声分布 P_n(w) 中采样 K 个负样本。
    常用的噪声分布是 unigram 分布的 3/4 次方：
        P_n(w) = (count(w)^{3/4}) / Σ_{i=1}^{V} (count(w_i)^{3/4})

    采样时，会从词汇表中按照上述分布独立采样 K 个词（不包括中心词 w_c）。
    实际实现中，通常使用随机采样的方式，每个词被选中的概率与其频率的3/4次方成正比。



5.2

In [11]:
import torch
import torch.nn.functional as F
import sys

torch.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize, precision=8, sci_mode=False, edgeitems=1000000)

def cbow_forward(context_indices, target_indices, W, W_out):
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 获取上下文词的嵌入向量
    context_embeds = W[context_indices]
    
    # 计算平均上下文向量（隐藏层）
    h = torch.mean(context_embeds, dim=1)
    
    # 计算输出得分
    scores = torch.matmul(h, W_out)
    
    # 计算softmax概率分布
    probs = F.softmax(scores, dim=1)
    
    # 计算交叉熵损失
    loss = F.cross_entropy(scores, target_indices)
    
    return loss, probs

def print_tensor_full(tensor, name):
    print(f"{name}:")
    print(tensor.tolist())


# 设置参数
torch.manual_seed(42)
V = 10
d = 5
batch_size = 4
context_size = 3

# 随机初始化
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)

# 随机生成上下文和目标
context_indices = torch.randint(0, V, (batch_size, context_size))
target_indices = torch.randint(0, V, (batch_size,))

print(f"V (词汇表大小): {V}")
print(f"d (嵌入维度): {d}")
print(f"batch_size: {batch_size}")
print(f"context_size: {context_size}")
print(f"context_indices 形状: {context_indices.shape}")
print(f"target_indices 形状: {target_indices.shape}")
print(f"W 形状: {W.shape}")
print(f"W_out 形状: {W_out.shape}")

print_tensor_full(context_indices, "context_indices")
print_tensor_full(target_indices, "target_indices")
print_tensor_full(W, "W 权重矩阵")
print_tensor_full(W_out, "W_out 权重矩阵")

# 前向传播
loss, probs = cbow_forward(context_indices, target_indices, W, W_out)

print(f"\n损失值: {loss.item():.8f}")
print_tensor_full(probs, "概率分布 (所有样本)")

# 验证梯度
loss.backward()
print(f"\nW 梯度形状: {W.grad.shape}")
print(f"W_out 梯度形状: {W_out.grad.shape}")
print_tensor_full(W.grad, "W 梯度")
print_tensor_full(W_out.grad, "W_out 梯度")

V (词汇表大小): 10
d (嵌入维度): 5
batch_size: 4
context_size: 3
context_indices 形状: torch.Size([4, 3])
target_indices 形状: torch.Size([4])
W 形状: torch.Size([10, 5])
W_out 形状: torch.Size([5, 10])
context_indices:
[[6, 0, 6], [8, 6, 8], [0, 6, 9], [0, 5, 9]]
target_indices:
[5, 2, 0, 8]
W 权重矩阵:
[[1.9269152879714966, 1.4872840642929077, 0.9007171988487244, -2.1055209636688232, 0.6784183979034424], [-1.2345448732376099, -0.04306747764348984, -1.6046669483184814, -0.7521352767944336, 1.6487230062484741], [-0.3924786448478699, -1.4036071300506592, -0.7278813123703003, -0.5594301819801331, -0.7688388824462891], [0.7624453902244568, 1.6423169374465942, -0.1595974713563919, -0.4973975419998169, 0.439589262008667], [-0.7581311464309692, 1.078317642211914, 0.8008005619049072, 1.680620551109314, 1.27912437915802], [1.2964228391647339, 0.610466480255127, 1.334737777709961, -0.2316243201494217, 0.041759490966796875], [-0.2515752911567688, 0.859858512878418, -1.3846737146377563, -0.8712361454963684, 0.0780238

6.1

In [21]:

print("""
给定 Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)，d_k = 4。

缩放点积注意力：
    Attention(Q, K, V) = softmax(Q * K^T / √d_k) * V

步骤1：计算得分矩阵 S = Q * K^T / √d_k
    Q: (2, 4), K^T: (4, 3) -> S: (2, 3)
    
    S[i,j] = (Q[i] · K[j]) / √4 = (Q[i] · K[j]) / 2

步骤2：对每一行（每个查询）应用 softmax，得到注意力权重 A
    A[i,j] = exp(S[i,j]) / Σ_{k=1}^{3} exp(S[i,k])
    A: (2, 3)

步骤3：加权求和得到输出 O = A * V
    O[i,:] = Σ_{j=1}^{3} A[i,j] * V[j,:]
    O: (2, 5)

具体数值计算（用符号表示所有元素）：

设 Q = [[q11, q12, q13, q14],
        [q21, q22, q23, q24]]
    K = [[k11, k12, k13, k14],
        [k21, k22, k23, k24],
        [k31, k32, k33, k34]]
    V = [[v11, v12, v13, v14, v15],
        [v21, v22, v23, v24, v25],
        [v31, v32, v33, v34, v35]]

步骤1：得分矩阵 S (2×3)
    S[0,0] = (q11*k11 + q12*k12 + q13*k13 + q14*k14) / 2
    S[0,1] = (q11*k21 + q12*k22 + q13*k23 + q14*k24) / 2
    S[0,2] = (q11*k31 + q12*k32 + q13*k33 + q14*k34) / 2
    S[1,0] = (q21*k11 + q22*k12 + q23*k13 + q24*k14) / 2
    S[1,1] = (q21*k21 + q22*k22 + q23*k23 + q24*k24) / 2
    S[1,2] = (q21*k31 + q22*k32 + q23*k33 + q24*k34) / 2

步骤2：softmax得到注意力权重 A (2×3)
    A[0,0] = exp(S[0,0]) / (exp(S[0,0]) + exp(S[0,1]) + exp(S[0,2]))
    A[0,1] = exp(S[0,1]) / (exp(S[0,0]) + exp(S[0,1]) + exp(S[0,2]))
    A[0,2] = exp(S[0,2]) / (exp(S[0,0]) + exp(S[0,1]) + exp(S[0,2]))
    A[1,0] = exp(S[1,0]) / (exp(S[1,0]) + exp(S[1,1]) + exp(S[1,2]))
    A[1,1] = exp(S[1,1]) / (exp(S[1,0]) + exp(S[1,1]) + exp(S[1,2]))
    A[1,2] = exp(S[1,2]) / (exp(S[1,0]) + exp(S[1,1]) + exp(S[1,2]))

步骤3：输出矩阵 O (2×5)
    O[0,0] = A[0,0]*v11 + A[0,1]*v21 + A[0,2]*v31
    O[0,1] = A[0,0]*v12 + A[0,1]*v22 + A[0,2]*v32
    O[0,2] = A[0,0]*v13 + A[0,1]*v23 + A[0,2]*v33
    O[0,3] = A[0,0]*v14 + A[0,1]*v24 + A[0,2]*v34
    O[0,4] = A[0,0]*v15 + A[0,1]*v25 + A[0,2]*v35
    O[1,0] = A[1,0]*v11 + A[1,1]*v21 + A[1,2]*v31
    O[1,1] = A[1,0]*v12 + A[1,1]*v22 + A[1,2]*v32
    O[1,2] = A[1,0]*v13 + A[1,1]*v23 + A[1,2]*v33
    O[1,3] = A[1,0]*v14 + A[1,1]*v24 + A[1,2]*v34
    O[1,4] = A[1,0]*v15 + A[1,1]*v25 + A[1,2]*v35

最终输出矩阵 O 的形状为 (2, 5)。
""")


给定 Q ∈ R^(2×4)，K ∈ R^(3×4)，V ∈ R^(3×5)，d_k = 4。

缩放点积注意力：
    Attention(Q, K, V) = softmax(Q * K^T / √d_k) * V

步骤1：计算得分矩阵 S = Q * K^T / √d_k
    Q: (2, 4), K^T: (4, 3) -> S: (2, 3)

    S[i,j] = (Q[i] · K[j]) / √4 = (Q[i] · K[j]) / 2

步骤2：对每一行（每个查询）应用 softmax，得到注意力权重 A
    A[i,j] = exp(S[i,j]) / Σ_{k=1}^{3} exp(S[i,k])
    A: (2, 3)

步骤3：加权求和得到输出 O = A * V
    O[i,:] = Σ_{j=1}^{3} A[i,j] * V[j,:]
    O: (2, 5)

具体数值计算（用符号表示所有元素）：

设 Q = [[q11, q12, q13, q14],
        [q21, q22, q23, q24]]
    K = [[k11, k12, k13, k14],
        [k21, k22, k23, k24],
        [k31, k32, k33, k34]]
    V = [[v11, v12, v13, v14, v15],
        [v21, v22, v23, v24, v25],
        [v31, v32, v33, v34, v35]]

步骤1：得分矩阵 S (2×3)
    S[0,0] = (q11*k11 + q12*k12 + q13*k13 + q14*k14) / 2
    S[0,1] = (q11*k21 + q12*k22 + q13*k23 + q14*k24) / 2
    S[0,2] = (q11*k31 + q12*k32 + q13*k33 + q14*k34) / 2
    S[1,0] = (q21*k11 + q22*k12 + q23*k13 + q24*k14) / 2
    S[1,1] = (q21*k21 + q22*k22 + q23*k23 + q24*k24) / 2
    

6.2

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys

# 强制完整显示
torch.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize, precision=8, sci_mode=False)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 重塑为多头形式
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        V = V.view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 2).transpose(1, 2)
        
        # 3. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        # 4. 拼接所有头的输出
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(seq_len, batch, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(attn_output)
        
        return output

def print_tensor_full(tensor, name):

    print(f"{name}:")
    print(tensor)

# 设置参数
torch.manual_seed(42)
d_model = 4
num_heads = 2
seq_len = 3
batch = 2

# 创建输入
X = torch.randn(seq_len, batch, d_model)

print_tensor_full(X, "输入 X")

# 创建多头注意力层
mha = MultiHeadAttention(d_model, num_heads)

# 前向传播
output = mha(X)

print(f"\n输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print_tensor_full(output[:, 0, :], "输出 (所有时间步, 第1个样本)")

# 验证形状
assert output.shape == X.shape
print("\n形状验证通过！输出形状与输入形状相同。")

# 查看参数
print(f"\n模型参数:")
total_params = sum(p.numel() for p in mha.parameters())
print(f"总参数: {total_params}")
print(f"\n各线性层参数:")
for name, param in mha.named_parameters():
    print(f"  {name}: 形状={param.shape}")
    print(f"    {param}")

输入 X:
tensor([[[ 1.92691529,  1.48728406,  0.90071720, -2.10552096],
         [ 0.67841840, -1.23454487, -0.04306748, -1.60466695]],

        [[ 0.35585991, -0.68662298, -0.49335635,  0.24148779],
         [-1.11090386,  0.09154566, -2.31692266, -0.21680473]],

        [[-0.30972677, -0.39571050,  0.80340934, -0.62159538],
         [-0.59200060, -0.06307438, -0.82855427,  0.33089843]]])

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
输出 (所有时间步, 第1个样本):
tensor([[-0.37890464, -0.20684996,  0.70771933, -0.45346010],
        [-0.19856676,  0.14263687,  0.30249754, -0.22081932],
        [-0.24297455, -0.30464020, -0.52796584, -0.54645759]], grad_fn=<SelectBackward0>)

形状验证通过！输出形状与输入形状相同。

模型参数:
总参数: 64

各线性层参数:
  W_q.weight: 形状=torch.Size([4, 4])
    Parameter containing:
tensor([[ 0.13434184, -0.13558972,  0.21042877,  0.44641107],
        [ 0.28902978, -0.21858627,  0.28863233,  0.08946311],
        [ 0.25391752, -0.30475253, -0.49495423, -0.19318026],
        [-0.38351142,  0.41